# 3.5 — ML mới trên folds chung (v2)

| Tên | Mô tả | Giữ lại khi |
|---|---|---|
| `lgb_nl3_cnt` | `lgb_nl3` + 13 feature đếm tần suất giá trị số (đếm trên train+test, không dùng nhãn) | OOF > 0.94774 (`lgb_nl3`) |
| `lgb_nl3_x3` | `lgb_nl3`, trung bình 3 seed (9, 19, 29) | gain > 0 |
| `lgb_nl4_x3` | `lgb_nl4`, trung bình 3 seed (10, 20, 30) | gain > 0 |
| `catboost_d4`, `catboost_d6` | CatBoost depth 4 / 6, category khai báo trực tiếp | gain > 0 |

Ô cuối đo **gain so với ensemble v1** (`ml+dl_all`, 5 ML + 4 DL): blend rank của ensemble v1 với model mới, trọng số chọn trên 4 fold, chấm trên fold còn lại.

**Input:** competition + `ccf-shared-code` (có `ml.py`) + output 3.0 + output 3.1 + output 3.3 (mới nhất). **Accelerator:** None (CPU).

In [ ]:
import sys
from pathlib import Path

# shared/ code: local repo or the Kaggle dataset that holds features.py + dl.py
cands = [Path("shared/features.py"), Path("../../shared/features.py"), *Path("/kaggle/input").rglob("features.py")]
feat = next(p for p in cands if p.is_file())
sys.path.insert(0, str(feat.parent.resolve()))
import features as F
print("shared code:", feat.parent)

In [ ]:
import time
import json
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

T0 = time.time()
def log(msg):
    print(f"[{(time.time() - T0) / 60:6.1f} min] {msg}", flush=True)

tr, te, y, train_id, test_id = F.load()
fold = F.load_folds(train_id)          # from notebook 3.0 output (folds_5_seed9.csv)
assert len(tr) == F.N_TRAIN and len(te) == F.N_TEST and set(np.unique(fold)) == set(range(F.N_FOLDS))
log(f"train {tr.shape}, test {te.shape}, fraud rate {y.mean():.4%}")

In [ ]:
import shutil
import ml as M

# Tách lần chạy: LGBM (CPU) -> MODELS = ["lgb_nl3_cnt", "lgb_nl3_x3", "lgb_nl4_x3"]
#                CatBoost (bật GPU T4 cho nhanh) -> MODELS = ["catboost_d4", "catboost_d6"]
# Không để một model xuất hiện ở hai notebook đã gắn làm input cho 3.6.
MODELS = ["lgb_nl3_cnt", "lgb_nl3_x3", "lgb_nl4_x3", "catboost_d4", "catboost_d6"]
CB_TASK = "GPU" if shutil.which("nvidia-smi") else "CPU"
print("MODELS:", MODELS, "| CatBoost task_type:", CB_TASK)

Xtr_t, Xte_t = F.tree_frame(tr), F.tree_frame(te)
Ctr, Cte = F.add_count_features(tr, te)
Xtr_cnt = pd.concat([Xtr_t, Ctr], axis=1)
Xte_cnt = pd.concat([Xte_t, Cte], axis=1)
print("count features:", list(Ctr.columns))
print(Ctr.describe().T[["mean", "min", "50%", "max"]].round(1))

Xtr_cb, Xte_cb = Xtr_t.copy(), Xte_t.copy()            # CatBoost: category dạng chuỗi
for c in F.CATS:
    Xtr_cb[c], Xte_cb[c] = Xtr_cb[c].astype(str), Xte_cb[c].astype(str)


def seed_avg(config, seeds):
    def run(a, b):
        o = p = 0
        its = []
        for s in seeds:
            oo, pp, it = M.fit_lgb(config, Xtr_t.iloc[a], y[a], Xtr_t.iloc[b], y[b], Xte_t, seed=s)
            o, p = o + oo / len(seeds), p + pp / len(seeds)
            its.append(it)
        return o, p, its
    return run


SPECS = {
    "lgb_nl3_cnt": lambda a, b: M.fit_lgb("lgb_nl3", Xtr_cnt.iloc[a], y[a], Xtr_cnt.iloc[b], y[b], Xte_cnt),
    "lgb_nl3_x3": seed_avg("lgb_nl3", [9, 19, 29]),
    "lgb_nl4_x3": seed_avg("lgb_nl4", [10, 20, 30]),
    "catboost_d4": lambda a, b: M.fit_catboost(Xtr_cb.iloc[a], y[a], Xtr_cb.iloc[b], y[b], Xte_cb, F.CATS, depth=4,
                                               task_type=CB_TASK),
    "catboost_d6": lambda a, b: M.fit_catboost(Xtr_cb.iloc[a], y[a], Xtr_cb.iloc[b], y[b], Xte_cb, F.CATS, depth=6,
                                               task_type=CB_TASK),
}

In [ ]:
all_metrics = {}
for name in MODELS:
    oof = np.zeros(len(tr))
    test_pred = np.zeros(len(te))
    iters = []
    for f, a, b in F.iter_folds(fold):
        t0 = time.time()
        o, p, it = SPECS[name](a, b)
        oof[b] = o
        test_pred += p / F.N_FOLDS
        iters.append(it)
        log(f"{name:12s} fold {f} | AUC={roc_auc_score(y[b], o):.5f} | iters={it} | {time.time() - t0:.0f}s")
    # CatBoost GPU lưu tên riêng (vd. catboost_d4_gpu) để chạy song song và so sánh với bản CPU
    out = name + ("_gpu" if name.startswith("catboost") and CB_TASK == "GPU" else "")
    all_metrics[out] = F.save_predictions(out, train_id, oof, test_id, test_pred, y, fold,
                                          extra={"source": "3.5", "iterations": iters, "task_type": CB_TASK})
    m = all_metrics[out]
    log(f"==> {out}: OOF AUC={m['oof_roc_auc']:.5f} | AP={m['oof_average_precision']:.5f} "
        f"| fold mean={m['fold_roc_auc_mean']:.5f}")

In [ ]:
# ---------- gain so với ensemble v1 (rank blend 5 ML + 4 DL, trọng số tối ưu trên toàn OOF)
from scipy.stats import spearmanr

V1 = ["lgb_nl3", "lgb_nl4", "lgb_nl8", "spline_lr", "xgb",
      "mlp_plr_sig3_reg_x3", "mlp_plr_sig1_reg", "mlp_plr_sig3", "mlp_plr_sig10_reg"]
avail = []
for n in V1:
    try:
        F.find_prediction("oof", n)
        avail.append(n)
    except FileNotFoundError:
        print("skip (không tìm thấy):", n)
O_ref, _ = F.load_predictions(avail, train_id, test_id)
R_ref = np.column_stack([F.rank01(O_ref[:, i]) for i in range(len(avail))])
w_ref, _ = F.tune_weights(R_ref, y, iters=500)
w_ref = 0.7 * w_ref + 0.3 / len(avail)
ref = F.rank01(R_ref @ w_ref)
lgb3 = O_ref[:, avail.index("lgb_nl3")]
print("reference ensemble:", avail)

rows = []
for n in all_metrics:
    o = F.load_predictions([n], train_id, test_id, roots=("3.ENSEMBLE",))[0][:, 0]
    base_auc, cv_auc, alphas = F.cv_alpha_gain(ref, F.rank01(o), y, fold)
    rows.append({"model": n, "oof_auc": roc_auc_score(y, o), "fold_auc_mean": all_metrics[n]["fold_roc_auc_mean"],
                 "spearman_vs_lgb_nl3": spearmanr(o, lgb3).correlation,
                 "ref_fold_auc": base_auc, "ref+model_cv_fold_auc": cv_auc, "gain": cv_auc - base_auc,
                 "alpha_per_fold": alphas})
cmp = pd.DataFrame(rows).set_index("model")
pd.set_option("display.width", 220)
print(cmp.round(5))
print("\nlgb_nl3 OOF AUC (3.1) =", round(roc_auc_score(y, lgb3), 5))
cmp.to_csv("3.ENSEMBLE/results/ml_new_vs_v1_3_5.csv")
log("done")